In [7]:
from fredapi import Fred
import pandas as pd
from config import FRED_API_KEY  # 见 T-A3 说明

fred = Fred(api_key=FRED_API_KEY)

# 联邦基金利率目标（有效利率，日度）
fedfunds = fred.get_series('FEDFUNDS', observation_start='1993-01-01')

# M2 货币供应量（月度，十亿美元）
m2 = fred.get_series('M2SL', observation_start='1993-01-01')

# PCE 通胀（月度，同比%）
pce = fred.get_series('PCEPI', observation_start='1993-01-01')
pce_yoy = pce.pct_change(12) * 100  # 转为同比增速

# 整理为 DataFrame
macro = pd.DataFrame({
    'fedfunds': fedfunds,
    'm2': m2,
    'pce_yoy': pce_yoy,
}).resample('ME').last()

macro.to_csv('data_raw/fed_rates_raw.csv')
print(macro.tail())

            fedfunds       m2   pce_yoy
2025-12-31      3.72  22353.6  2.878084
2026-01-31      3.64  22429.3  2.856869
2026-02-28      3.64  22627.3  2.829552
2026-03-31      3.64  22686.0  3.496081
2026-04-30      3.64      NaN       NaN


In [1]:
%pip install fredapi yfinance pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import yfinance as yf

# 各类资产代码说明
assets = {
    '^GSPC':  'SP500',       # 标普 500 指数
    '^NDX':   'Nasdaq100',   # 纳斯达克 100
    'GLD':    'Gold_ETF',    # 黄金 ETF（SPDR）
    'TLT':    'TBond_20Y',   # 20 年期美债 ETF
    'BTC-USD':'Bitcoin',     # 比特币（2014 年起有数据）
    'DX-Y.NYB':'DXY',        # 美元指数
    '^VIX':   'VIX',         # 恐慌指数
}

prices = yf.download(
    list(assets.keys()),
    start='1993-01-01',
    auto_adjust=True
)['Close']

prices.columns = [assets[c] for c in prices.columns]
prices_monthly = prices.resample('ME').last()
prices_monthly.to_csv('data_raw/assets_raw.csv')
print(prices_monthly.tail())

[*********************100%***********************]  7 of 7 completed

                 Bitcoin        DXY    Gold_ETF  TBond_20Y        SP500  \
Date                                                                      
2026-01-31  78621.117188  96.989998  444.950012  85.849030  6939.029785   
2026-02-28  66995.859375  97.610001  483.750000  89.827065  6878.879883   
2026-03-31  68233.312500  99.959999  430.290009  86.027336  6528.520020   
2026-04-30  76304.320312  98.080002  423.660004  85.305000  7209.009766   
2026-05-31  81129.937500  98.469002  432.929993  84.989998  7400.959961   

               Nasdaq100        VIX  
Date                                 
2026-01-31  25552.390625  17.440001  
2026-02-28  24960.039062  19.860001  
2026-03-31  23740.189453  25.250000  
2026-04-30  27452.119141  16.889999  
2026-05-31  29064.800781  17.850000  
